In [ ]:
import cv2
import pygame
import random
import os
import glob
import time
import mediapipe as mp

# Carpetas de imágenes y audios
CARPETA_FOTOS = "fotos"
CARPETA_AUDIOS = "audios"

# Configuración de la ruleta
DURACION_RULETA = 4.0
VELOCIDAD_RULETA = 0.12

# Inicializa el reproductor de audio
pygame.mixer.init()

# Activa la cámara
cap = cv2.VideoCapture(0)

# Inicializa MediaPipe Face Mesh
mp_face_mesh = mp.solutions.face_mesh
face_mesh = mp_face_mesh.FaceMesh(max_num_faces=1)

archivos_fotos = []
archivos_audios = {}
elementos = []

# Busca todas las imágenes disponibles
for ext in ["*.png", "*.jpg", "*.jpeg", "*.webp"]:
    archivos_fotos.extend(glob.glob(os.path.join(CARPETA_FOTOS, ext)))

# Guarda los audios asociados a cada imagen
for archivo in glob.glob(os.path.join(CARPETA_AUDIOS, "*.wav")):
    nombre = os.path.splitext(os.path.basename(archivo))[0]
    archivos_audios[nombre] = archivo

# Carga las imágenes y relaciona cada una con su audio
for foto in archivos_fotos:
    nombre = os.path.splitext(os.path.basename(foto))[0]
    imagen = cv2.imread(foto, cv2.IMREAD_UNCHANGED)

    if imagen is not None:
        elementos.append({
            "imagen": imagen,
            "audio": archivos_audios.get(nombre)
        })

# Comprueba que existen imágenes
if not elementos:
    raise Exception("No hay fotografias validas")

# Variables del estado de la aplicación
cara = None
ausencias = 0
activado = False
landmarks = False

ruleta_activa = False
selecciona = None

indice_actual = 0

tiempo_inicio = 0
ultimo_cambio = 0

imagen_ruleta = elementos[0]["imagen"]

# Detecta landmarks
def detectar_landmarks(frame):

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    resultados = face_mesh.process(rgb)

    if resultados.multi_face_landmarks:
        return resultados.multi_face_landmarks[0].landmark

    return None

# Detecta la posición de la cara a partir de los landmarks
def obtener_cara(frame, landmarks):

    if landmarks is None:
        return None

    xs = [p.x for p in landmarks]
    ys = [p.y for p in landmarks]

    x1 = int(min(xs) * frame.shape[1])
    y1 = int(min(ys) * frame.shape[0])

    x2 = int(max(xs) * frame.shape[1])
    y2 = int(max(ys) * frame.shape[0])

    return (x1, y1, x2 - x1, y2 - y1)

# Dibuja landmarks detectados en la cara
def dibujar_landmarks(frame, landmarks):

    if landmarks is not None:
        for punto in landmarks:

            x = int(punto.x * frame.shape[1])
            y = int(punto.y * frame.shape[0])

            cv2.circle(frame, (x, y), 2, (0, 255, 0), -1)

    return frame

# Inicia la ruleta de imágenes
def iniciar_ruleta():

    global ruleta_activa
    global tiempo_inicio
    global ultimo_cambio
    global selecciona

    ruleta_activa = True
    selecciona = None

    tiempo_inicio = time.monotonic()
    ultimo_cambio = tiempo_inicio

# Reinicia el filtro y sus variables
def reset():

    global cara
    global activado
    global selecciona
    global ruleta_activa
    global ausencias

    cara = None
    activado = False
    selecciona = None
    ruleta_activa = False
    ausencias = 0

# Finaliza la ruleta y reproduce el audio correspondiente
def finalizar_ruleta():

    global selecciona
    global ruleta_activa

    selecciona = elementos[indice_actual]
    ruleta_activa = False

    audio = selecciona["audio"]

    if audio:
        try:
            pygame.mixer.music.stop()
            pygame.mixer.music.load(audio)
            pygame.mixer.music.play()

        except pygame.error as e:
            print("Error de audio:", e)


# Coloca la imagen sobre la posición de la cara
def colocar_imagen(frame, imagen, cara):

    x, y, w, h = cara

    # Calcula el tamaño de la imagen
    nuevo_ancho = int(w * 0.7)
    nuevo_alto = int(imagen.shape[0] * nuevo_ancho / imagen.shape[1])

    nuevo_alto = min(nuevo_alto, int(h * 0.4))
    nuevo_ancho = int(imagen.shape[1] * nuevo_alto / imagen.shape[0])

    # Redimensiona la imagen
    img = cv2.resize(imagen, (nuevo_ancho, nuevo_alto), interpolation=cv2.INTER_LINEAR)

    # Calcula la posición centrada sobre la cara
    centro_x = x + w // 2
    centro_y = y - int(h * 0.10)

    inicio_x = centro_x - nuevo_ancho // 2
    inicio_y = centro_y - nuevo_alto // 2

    # Evita que la imagen salga de los límites del vídeo
    x1 = max(0, inicio_x)
    y1 = max(0, inicio_y)
    x2 = min(frame.shape[1], inicio_x + nuevo_ancho)
    y2 = min(frame.shape[0], inicio_y + nuevo_alto)

    if x1 >= x2 or y1 >= y2:
        return frame

    # Calcula el recorte correspondiente
    ix1 = x1 - inicio_x
    iy1 = y1 - inicio_y
    ix2 = ix1 + x2 - x1
    iy2 = iy1 + y2 - y1

    recorte = img[iy1:iy2, ix1:ix2]
    fondo = frame[y1:y2, x1:x2]

    # Combina la transparencia si la imagen tiene canal alfa
    if recorte.shape[2] == 4:

        alpha = recorte[:, :, 3:4].astype("float32") / 255.0
        frame[y1:y2, x1:x2] = (alpha * recorte[:, :, :3] + (1 - alpha) * fondo).astype("uint8")

    else:
        frame[y1:y2, x1:x2] = recorte

    return frame

# Bucle principal de la cámara
while cap.isOpened():

    ret, frame = cap.read()

    if not ret:
        break

    # Calcula los landmarks
    landmarks_actuales = detectar_landmarks(frame)

    # Obtiene el rectángulo de la cara
    nueva_cara = obtener_cara(frame, landmarks_actuales)

    if nueva_cara is not None:

        cara = nueva_cara
        ausencias = 0

        # Inicia la ruleta al detectar una cara por primera vez
        if not activado:
            iniciar_ruleta()
            activado = True

    else:
        ausencias += 1

        # Reinicia si se pierde la cara durante varios fotogramas
        if ausencias >= 5:
            reset()

    # Actualiza la ruleta mientras está activa
    if ruleta_activa:

        ahora = time.monotonic()

        # Cambia aleatoriamente la imagen.
        if ahora - ultimo_cambio >= VELOCIDAD_RULETA:

            indice_actual = random.randrange(len(elementos))
            imagen_ruleta = elementos[indice_actual]["imagen"]
            ultimo_cambio = ahora

        # Finaliza la ruleta cuando termina el tiempo
        if ahora - tiempo_inicio >= DURACION_RULETA:
            finalizar_ruleta()

    # Prepara el fotograma de salida
    salida = frame.copy()

    if cara is not None:

        # Selecciona qué imagen debe mostrarse
        if ruleta_activa:
            imagen = imagen_ruleta

        elif selecciona is not None:
            imagen = selecciona["imagen"]

        else:
            imagen = None

        # Dibuja el filtro sobre la cara
        if imagen is not None:
            salida = colocar_imagen(salida, imagen, cara)

    # Muestra los landmarks
    if landmarks:
        salida = dibujar_landmarks(salida, landmarks_actuales)
    
    # Muestra el resultado de la cámara
    cv2.imshow("WINDOWS FILTER", salida)

    # Lee el teclado
    tecla = cv2.waitKey(1) & 0xFF

    # esc: cierra el programa
    if tecla == 27:
        break
    # r: reinicia el filtro
    elif tecla == ord('r'):
        reset()
    elif tecla == ord('f'):
        landmarks = not landmarks

# Libera los recursos al cerrar el programa
cap.release()
face_mesh.close()

cv2.destroyAllWindows()

pygame.mixer.music.stop()
pygame.mixer.quit()
